# Fundamentos del proyecto

Primera iteración de Sergio: configuración, acciones y estructuras inmutables.
Los ejemplos son **sintéticos**; todavía no hay simulación de episodios ni entrenamiento.

Reinicia el kernel y ejecuta todas las celdas. Consulta [primeros pasos](../docs/PRIMEROS_PASOS.md)
y el seguimiento en [TODO](../docs/TODO.md).

## Comprobar el entorno

Instala primero `uv sync --locked --group notebooks`. La raíz se detecta desde la carpeta
actual sin alterar las rutas de importación. CPU basta para este recorrido.

In [ ]:
import sys
from dataclasses import fields
from importlib.metadata import version
from pathlib import Path

import jax
import jax.numpy as jnp
from IPython.display import Markdown, display

from darkforest.config import ACTIONS, EnvConfig, load_config
from darkforest.env import DarkForestEnv
from darkforest.types import Obs

candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((path for path in candidates if (path / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abre el notebook dentro del repositorio RL-DarkForest")
print("Raíz:", ROOT)
print("Python:", sys.version.split()[0])
for package in ("jax", "flax", "jupyterlab", "ipykernel", "nbconvert"):
    print(f"{package}: {version(package)}")
print("Dispositivos:", jax.devices())

## Cargar configuración y overrides

El YAML de H1 hereda la base. Los overrides prevalecen sin modificar el archivo ni la
configuración previa. Letalidad cero configura el caso H1; no evalúa la hipótesis.

In [ ]:
config_path = ROOT / "configs" / "h1_sanity.yaml"
cfg = load_config(config_path)
assert cfg.env.n_civs == 2
assert cfg.env.anomaly.lethality == 0.0
assert cfg.env.n_discrete == 13
custom = load_config(config_path, {"train.seed": 7, "env.max_steps": 30})
assert custom.train.seed == 7
assert custom.env.max_steps == 30
assert cfg == load_config(config_path)
print("Civilizaciones:", cfg.env.n_civs)
print("Letalidad:", cfg.env.anomaly.lethality)
print("Overrides:", custom.train.seed, custom.env.max_steps)

## Inspeccionar las acciones

Siete familias producen 13 IDs con cuatro símbolos y cuatro ranuras. Transmitir recibe
un símbolo y atacar una ranura, ambos desde cero; `-1` indica ausencia de argumento.

In [ ]:
env = DarkForestEnv(cfg.env)
actions = jnp.arange(cfg.env.n_discrete, dtype=jnp.int32)
families, arguments = env.decode_action(actions)
rows = ["| ID | Familia | Argumento |", "|---|---|---|"]
for action, family, argument in zip(
    actions.tolist(), families.tolist(), arguments.tolist(), strict=True
):
    rows.append(f"| {action} | {ACTIONS[family]} | {argument} |")
display(Markdown("\n".join(rows)))
assert len(rows) - 2 == 13

In [ ]:
compiled = jax.jit(env.decode_action)(actions)
vectorized = jax.vmap(env.decode_action)(actions)
for expected, actual_jit, actual_vmap in zip(
    (families, arguments), compiled, vectorized, strict=True
):
    assert bool(jnp.array_equal(expected, actual_jit))
    assert bool(jnp.array_equal(expected, actual_vmap))
invalid = env.decode_action(jnp.array([-1, cfg.env.n_discrete], dtype=jnp.int32))
assert all(bool(jnp.all(part == -1)) for part in invalid)
print("Normal, jit y vmap coinciden; IDs inválidos: (-1, -1).")

## Reconocer errores

`KeyError`: clave desconocida. `TypeError`: tipo incorrecto. `ValueError`: valor inválido.
Las velocidades incompatibles se rechazan sin ajustes automáticos. Seeds: `0..4294967295`.
Las celdas siguientes capturan los errores esperados y permiten continuar.

In [ ]:
invalid_configs = [
    ({"env.n_civs": 3}, ValueError),
    ({"env.n_civs": 2.0}, TypeError),
    ({"env.proj_speed": 0.2}, ValueError),
    ({"env.p_kill": 1.1}, ValueError),
    ({"env.growth": float("nan")}, ValueError),
    ({"train.seed": -1}, ValueError),
    ({"env.parametro_desconocido": 1}, KeyError),
]
for overrides, error_type in invalid_configs:
    try:
        load_config(config_path, overrides)
    except error_type as error:
        print(f"{error_type.__name__}: {error}")
    else:
        raise AssertionError(f"Se esperaba {error_type.__name__}: {overrides}")
try:
    EnvConfig(n_civs=3)
except ValueError as error:
    print("Construcción directa:", error)
else:
    raise AssertionError("La dataclass también debe validar n_civs")
try:
    env.decode_action(jnp.array(1.5))
except TypeError as error:
    print("Tipo de acción:", error)
else:
    raise AssertionError("Una acción decimal debe rechazarse")

## Observaciones y máscaras sintéticas

Dos agentes con una característica y una medición ilustrativas. **Estas dimensiones no
fijan el diseño físico pendiente.** `False` indica relleno; 42 es un dato artificial válido.
`Obs` no contiene θ ni diagnósticos del estado completo.

In [ ]:
n_agents = cfg.env.n_civs
n_slots = cfg.env.max_detections
obs = Obs(
    own=jnp.array([[10.0, 0.0], [12.0, 1.0]], dtype=jnp.float32),
    detections=jnp.zeros((n_agents, n_slots, 1)).at[0, 0, 0].set(42.0),
    det_mask=jnp.zeros((n_agents, n_slots), dtype=bool).at[0, 0].set(True),
    own_meas=jnp.zeros((n_agents, 1)),
    since_burst=jnp.zeros(n_agents, dtype=jnp.int32),
    alive=jnp.ones(n_agents, dtype=bool),
    agent_id=jnp.arange(n_agents, dtype=jnp.int32),
)
assert "theta" not in {field.name for field in fields(Obs)}
assert int(obs.det_mask.sum()) == 1
visible = jnp.where(obs.det_mask[..., None], obs.detections, 0.0)
assert float(visible.sum()) == 42.0
print("Máscaras:", obs.det_mask)
print("Formas del pytree:", [leaf.shape for leaf in jax.tree.leaves(obs)])

In [ ]:
updated = obs.replace(own=obs.own.at[0, 0].set(15.0))
assert float(obs.own[0, 0]) == 10.0
assert float(updated.own[0, 0]) == 15.0
assert updated is not obs
assert jax.tree.structure(updated) == jax.tree.structure(obs)
print("Recursos originales:", obs.own[:, 0])
print("Recursos de la nueva estructura:", updated.own[:, 0])

## Cambiar parámetros

El ejercicio usa dos símbolos y mantiene cuatro ranuras: deben resultar 11 acciones.
Después puedes cargar otra configuración con seis civilizaciones y una seed válida distinta.
El ejemplo sintético anterior se mantiene para dos agentes. Reinicia el kernel y ejecuta todo.

In [ ]:
exercise = load_config(config_path, {"env.n_symbols": 2})
exercise_env = DarkForestEnv(exercise.env)
assert exercise.env.n_discrete == 11
last_family, last_argument = exercise_env.decode_action(exercise.env.n_discrete - 1)
assert (int(last_family), int(last_argument)) == (6, -1)
assert cfg.env.n_symbols == 4
print("Acciones con dos símbolos:", exercise.env.n_discrete)
print("Fundamentos comprobados; el simulador se construirá después.")

## Lo siguiente

Cerrar dimensiones, colas y refugio; implementar episodios, políticas, guardado y reproducción.
Luego una GRU compartida, rollouts, GAE y PPO permitirán aprender. El autojuego añadirá
oponentes congelados y la evaluación producirá los archivos de análisis.

No hay premios extra por cooperar o atacar. Consulta [TAREAS](../docs/TAREAS.md) y actualiza
[TODO](../docs/TODO.md). Limpia las salidas antes de versionar cambios en el notebook.